In [2]:
import pandas as pd
import seaborn as sns
import matplotlib
from datetime import datetime
import requests
import json
import numpy as np
import ast
import re
from collections import Counter
import datetime
import os
import logging
import traceback
import time
import openai

# This Script is intended to run 1 time
### to manually label movies that chatGPT was not able to label itself

In [3]:
labeled_movies = pd.read_csv("wiki_movies_mood_label_by_gpt-3.5-turbo.csv", low_memory=False)

In [1]:
from openai import OpenAI

client = OpenAI(
    # defaults to os.environ.get("OPENAI_API_KEY")
    api_key="INSERT KEY HERE",
)

In [4]:
def label_movies(idx,happy,sad,energetic,calm):
    labeled_movies.at[idx,'happy']=happy
    labeled_movies.at[idx,'sad']=sad
    labeled_movies.at[idx,'energetic']=energetic
    labeled_movies.at[idx,'calm']=calm
    labeled_movies.at[idx,'error_labeling']=''

In [34]:
def generate_prompt(row):
    if row.genres and row.new_plot:
        prompt = 'Based on a movie with genres "'+' and'.join(ast.literal_eval(row.genres)) + \
                 '" and an overall plot "' + row.new_plot + \
                 '" rank each mood happy, sad, energetic, calm on a scale from 0.00 to 1.00 like' \
                 ' this Happy: x.xx Sad: x.xx Energetic: x.xx Calm: x.xx'
        # Make API Call
        happy,sad,energetic,calm,err_d = openAPI_call(prompt, row)
    elif row.new_plot:
        #prompt = 'Based on this movie with a plot "' + row['new_plot']+\
        #'" rank each mood happy, sad, energetic, calm on a scale from 0.00 to 1.00.'
        prompt = 'Based on a movie with an overall plot "' + row['new_plot'] + \
                 '" rank each mood happy, sad, energetic, calm on a scale from 0.00 to 1.00 like' \
                 'this Happy: x.xx Sad: x.xx Energetic: x.xx Calm: x.xx'
        # Make API Call\
        happy,sad,energetic,calm,err_d = openAPI_call(prompt, row)
    else:
        print("--------------------")
        print("Insufficient data: ")
        print("movie title: "+ row.title)
        print("new_plot: "+ row.new_plot)
        print("genres: "+ row.genres)
        print("--------------------")
        return (0.0,0.0,0.0,0.0,{'error' : True, 'happy' : None, 'sad' : \
                                 None, 'energetic' : None, 'calm' : None \
                                }
               )

    return (happy,sad,energetic,calm,err_d)

In [41]:
def openAPI_call(prompt, row):
    happy_error     = False
    sad_error       = False
    energetic_error = False
    calm_error      = False
    completion = client.chat.completions.create(
                  model="gpt-3.5-turbo",
                  messages=[
                    {"role": "user", "content": prompt}
                  ],
                  temperature=0.5,
                )
    '''
    completion = chat_completion_with_backoff(
                    model="gpt-3.5-turbo",
                    messages=[
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.5
                 )
    '''

    content = completion.choices[0].message.content
    content = content.lower()

    try:
        happy = re.search('happy: \d.\d', content).group()
        happy = re.search('\d.\d', happy).group()
        happy = float(happy)
    except AttributeError:
        happy = 0.0
        happy_error = True
    
    try:
        sad = re.search("sad: \d.\d", content).group()
        sad = re.search('\d.\d', sad).group()
        sad = float(sad)
    except AttributeError:
        sad = 0.0
        sad_error = True
        
    try:
        energetic = re.search("energetic: \d.\d", content).group()
        energetic = re.search('\d.\d', energetic).group()
        energetic = float(energetic)
    except AttributeError:
        energetic = 0.0
        energetic_error = True
        
    try:
        calm = re.search("calm: \d.\d", content).group()
        calm = re.search('\d.\d', calm).group()
        calm = float(calm)
    except AttributeError:
        calm = 0.0
        calm_error = True
    
    if happy_error or sad_error or energetic_error or calm_error:
        print("--------------------")
    if happy_error:
        print("No Happy Score Found")
    if sad_error:
        print("No Sad Score Found")
    if energetic_error:
        print("No Energetic Score Found")
    if calm_error:
        print("No Calm Score Found")
    
    if happy_error or sad_error or energetic_error or calm_error:
        print("prompt: " + prompt)
        print("GPT response: " + content)
        print("\n")
        print("movie title: "+ row.title)
        print("new_plot: "+ row.new_plot)
        print("genres: "+ row.genres)
        
        print("--------------------")
    error_occured = True if happy_error or sad_error or energetic_error or calm_error else False
    return happy, sad, energetic, calm, {'error' : error_occured, 'happy' : happy_error, 'sad' : \
                                         sad_error, 'energetic' : energetic_error, 'calm' : calm_error \
                                        }

In [ ]:
labeled_movies[labeled_movies.title == 'August Underground'] # index 6214 DONE
labeled_movies[labeled_movies.title == 'Man Bites Dog (film)'] # index 3861 DONE
labeled_movies[labeled_movies.title == 'Super Size Me'] # index 7017 DONE
labeled_movies[labeled_movies.title == 'My Little Pony: A Very Pony Place'] # index 8249 DONE
labeled_movies[labeled_movies.title == 'Human Nature (2001 film)'] # index 6297 DONE
labeled_movies[labeled_movies.title == 'The Search for Signs of Intelligent Life in the Universe'] # index 3705 DONE
labeled_movies[labeled_movies.title == 'A Christmas Carol'] # index 2923 DONE
labeled_movies[labeled_movies.title == 'Never Cry Wolf (film)'] # index 1687 DONE
labeled_movies[labeled_movies.title == 'Chariots of Fur'] # index 4246 DONE
labeled_movies[labeled_movies.title == 'Powaqqatsi'] # index 2893 DONE
labeled_movies[labeled_movies.title == 'Honky Tonk Freeway'] # index 1356 DONE

In [7]:
label_movies(6214,0.0,1.0,0.0,0.0) # August Underground

In [8]:
labeled_movies[labeled_movies.title == 'August Underground'] # index 6214

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
6214,215,6553,2001,August_Underground,August Underground,"Peter, a serial killer, invites his camera-wie...",NaN,NaN,Directed byFred Vogel,NaN,...,NaN,0,0,0,0,0.0,1.0,0.0,0.0,


In [9]:
label_movies(3861,0.0,1.0,0.0,0.0) # Man Bites Dog (film)

In [10]:
labeled_movies[labeled_movies.title == 'Man Bites Dog (film)'] # index 3861

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
3861,861,4069,1992,Man_Bites_Dog_(film),Man Bites Dog (film),Ben is a witty and charismatic but narcissisti...,NaN,NaN,Directed byRémy BelvauxAndré BonzelBenoît Poel...,Screenplay byRémy BelvauxAndré BonzelBenoît Po...,...,NaN,0,0,1,0,0.0,1.0,0.0,0.0,


In [11]:
label_movies(7017,0.3,0.6,0.2,0.5) # Super Size Me

In [12]:
labeled_movies[labeled_movies.title == 'Super Size Me'] # index 7017

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
7017,1018,7443,2004,Super_Size_Me,Super Size Me,NaN,NaN,"As the film begins, Spurlock is in above avera...",Directed byMorgan Spurlock,NaN,...,341.538461,1,2,0,2,0.3,0.6,0.2,0.5,


In [13]:
label_movies(8249,0.8,0.1,0.5,0.2) # My Little Pony: A Very Pony Place

In [14]:
labeled_movies[labeled_movies.title == 'My Little Pony: A Very Pony Place'] # index 8249

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
8249,2250,8713,2009,My_Little_Pony:_A_Very_Pony_Place,My Little Pony: A Very Pony Place,"Unlike the previous three specials, A Very Pon...",NaN,NaN,Directed byJohn Grusd,NaN,...,NaN,0,0,0,0,0.8,0.1,0.5,0.2,


In [15]:
label_movies(6297,0.2,0.70,0.10,0.50) # Human Nature (2001 film)

In [16]:
labeled_movies[labeled_movies.title == 'Human Nature (2001 film)'] # index 6297

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
6297,298,6647,2001,Human_Nature_(2001_film),Human Nature (2001 film),Three characters are recounting events from th...,NaN,NaN,Directed byMichel Gondry,NaN,...,0.186047,0,0,0,0,0.2,0.7,0.1,0.5,


In [17]:
label_movies(3705,0.80,0.20,0.70,0.40) # The Search for Signs of Intelligent Life in the Universe

In [18]:
labeled_movies[labeled_movies.title == 'The Search for Signs of Intelligent Life in the Universe'] # index 3705

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
3705,705,3909,1991,The_Search_for_Signs_of_Intelligent_Life_in_th...,The Search for Signs of Intelligent Life in th...,NaN,NaN,NaN,Directed byJohn Bailey,NaN,...,NaN,0,0,0,0,0.8,0.2,0.7,0.4,


In [19]:
label_movies(2923,0.30,0.70,0.20,0.50) # A Christmas Carol

In [20]:
labeled_movies[labeled_movies.title == 'A Christmas Carol'] # index 2923

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
2923,2923,3075,1988,A_Christmas_Carol,A Christmas Carol,"The book is divided into five chapters, which ...",NaN,NaN,NaN,NaN,...,NaN,0,0,0,0,0.3,0.7,0.2,0.5,


In [21]:
label_movies(1687,0.30,0.40,0.20,0.70) # Never Cry Wolf (film)

In [22]:
labeled_movies[labeled_movies.title == 'Never Cry Wolf (film)'] # index 1687

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
1687,1687,1774,1983,Never_Cry_Wolf_(film),Never Cry Wolf (film),"The young, naïve Canadian biologist Tyler is a...",NaN,NaN,Directed byCarroll Ballard,Screenplay byCurtis HansonSam HammRichard Kletter,...,2.509091,1,1,1,0,0.3,0.4,0.2,0.7,


In [23]:
label_movies(4246,0.80,0.10,0.90,0.20) # Chariots of Fur

In [24]:
labeled_movies[labeled_movies.title == 'Chariots of Fur'] # index 4246

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
4246,1246,4471,1994,Chariots_of_Fur,Chariots of Fur,"Introduction: At the beginning, we see the Roa...",NaN,NaN,Directed byChuck Jones,NaN,...,NaN,0,0,0,0,0.8,0.1,0.9,0.2,


In [25]:
label_movies(2893,0.30,0.50,0.40,0.70) # Powaqqatsi

In [26]:
labeled_movies[labeled_movies.title == 'Powaqqatsi'] # index 2893

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
2893,2893,3043,1988,Powaqqatsi,Powaqqatsi,NaN,NaN,"In the beginning chapter, Serra Pelada, men fr...",Directed byGodfrey Reggio,NaN,...,NaN,0,0,0,0,0.3,0.5,0.4,0.7,


In [27]:
label_movies(1356,0.70,0.20,0.60,0.30) # Honky Tonk Freeway

In [28]:
labeled_movies[labeled_movies.title == 'Honky Tonk Freeway'] # index 1356

,Unnamed: 0.1,Unnamed: 0,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
1356,1356,1427,1981,Honky_Tonk_Freeway,Honky Tonk Freeway,"In a small Florida tourist town named Ticlaw, ...",NaN,NaN,Directed byJohn Schlesinger,NaN,...,0.083333,0,0,0,0,0.7,0.2,0.6,0.3,


## Now that we are done labeling the movies that chatGPT had issues with, lets focus on the movies that OpenAI was down for 

In [30]:
unlabeled_movies_remaining = labeled_movies[(labeled_movies.happy == 0) & (labeled_movies.sad == 0) & (labeled_movies.energetic == 0) & (labeled_movies.calm == 0)]

In [31]:
unlabeled_movies_remaining.shape

(37, 55)

## Out of these 37 unlabeled movies, we know 4 are the highly explicit, lets focus on the ones that aren't

In [ ]:
# OpenAI Outage
#labeled_movies[labeled_movies.title == 'Never Forever'] # index 7868 DONE
#labeled_movies[labeled_movies.title == 'Prom Night (2008 film)'] # index 7869 DONE
#labeled_movies[labeled_movies.title == 'Smart People'] # index 7870 DONE
#labeled_movies[labeled_movies.title == 'Street Kings'] # index 7871 DONE
#labeled_movies[labeled_movies.title == 'The Visitor (2007 drama film)'] # index 7872 DONE
#labeled_movies[labeled_movies.title == '88 Minutes'] # index 7873 DONE
#labeled_movies[labeled_movies.title == 'Forgetting Sarah Marshall'] # index 7877 DONE
#labeled_movies[labeled_movies.title == 'Where in the World Is Osama bin Laden?'] # index 7879 DONE
#labeled_movies[labeled_movies.title == 'Young@Heart (film)'] # index 7880 DONE
#labeled_movies[labeled_movies.title == 'Baby Mama (film)'] # index 7882 DONE
#labeled_movies[labeled_movies.title == 'Deal (2008 film)'] # index 7883 DONE
#labeled_movies[labeled_movies.title == 'Deception (2008 film)'] # index 7884 DONE
#labeled_movies[labeled_movies.title == 'Harold & Kumar Escape from Guantanamo Bay'] # index 7885 DONE
#labeled_movies[labeled_movies.title == 'Mister Lonely'] # index 7888 DONE
#labeled_movies[labeled_movies.title == 'Redbelt'] # index 7890 DONE
#labeled_movies[labeled_movies.title == 'The Fall (2006 film)'] # index 7893 DONE
#labeled_movies[labeled_movies.title == 'And When Did You Last See Your Father?'] # index 7906 DONE
#labeled_movies[labeled_movies.title == 'You Don\'t Mess with the Zohan'] # index 7909 DONE
#labeled_movies[labeled_movies.title == 'Encounters at the End of the World'] # index 7910 DONE
#labeled_movies[labeled_movies.title == 'The Children of Huang Shi'] # index 7911 DONE
#labeled_movies[labeled_movies.title == 'The Happening (2008 film)'] # index 7912 DONE
#labeled_movies[labeled_movies.title == 'The Incredible Hulk (film)'] # index 7913 DONE
#labeled_movies[labeled_movies.title == 'Quid Pro Quo (film)'] # index 7914 DONE
#labeled_movies[labeled_movies.title == 'Brick Lane (2007 film)'] # index 7915 DONE
#labeled_movies[labeled_movies.title == 'Get Smart (film)'] # index 7916 DONE
#labeled_movies[labeled_movies.title == 'Kit Kittredge: An American Girl'] # index 7917 DONE
#labeled_movies[labeled_movies.title == 'The Love Guru'] # index 7918 DONE
#labeled_movies[labeled_movies.title == 'Wanted (2008 film)'] # index 7920 DONE
#labeled_movies[labeled_movies.title == 'Tell No One'] # index 7921 DONE
#labeled_movies[labeled_movies.title == 'The Wackness'] # index 7922 DONE
#labeled_movies[labeled_movies.title == 'Garden Party (2008 film)'] # index 7923 DONE
#labeled_movies[labeled_movies.title == 'Hellboy II: The Golden Army'] # index 7924 DONE
#labeled_movies[labeled_movies.title == 'Finding Amanda'] # index 7919 DONE

In [42]:
# list of indices to make call to openAPI
# iterate through each, and make openAPI call
unlabeled_movie_lst = [7868,7869,7870,7871,7872,7873,7877,7879,7880,7882,7883,7884,7885,7888,7890,7893,7906,\
                       7909,7910,7911,7912,7913,7914,7915,7916,7917,7918, 7919,7920,7921,7922,7923,7924 \
                      ]
counter = 0
for index in unlabeled_movie_lst:
    if labeled_movies.at[index,'happy'] == 0 and labeled_movies.at[index,'sad'] == 0 and \
    labeled_movies.at[index,'energetic'] == 0 and labeled_movies.at[index,'calm'] == 0:
        try:
            happy,sad,energetic,calm,err_d = generate_prompt(labeled_movies.iloc[index])
        except Exception as e:
            print(traceback.format_exc())
            continue
        labeled_movies.at[index, 'happy'] = happy
        labeled_movies.at[index, 'sad'] = sad
        labeled_movies.at[index, 'energetic'] = energetic
        labeled_movies.at[index, 'calm'] = calm
        if err_d['error']:
            error_string = ''
            if err_d['happy']:
                error_string += "No Happy Score Found "
            if err_d['sad']:
                error_string += "No Sad Score Found "
            if err_d['energetic']:
                error_string += "No Energetic Score Found "
            if err_d['calm']:
                error_string += "No Calm Score Found "
            labeled_movies.at[row.Index, 'error_labeling'] = error_string
        counter += 1
        print("completed " + str(counter) + " rows")
    else:
        print('movie is labeled')

completed 1 rows
completed 2 rows
completed 3 rows
completed 4 rows
completed 5 rows
completed 6 rows
completed 7 rows
completed 8 rows
completed 9 rows
completed 10 rows
completed 11 rows
completed 12 rows
completed 13 rows
completed 14 rows
completed 15 rows
completed 16 rows
completed 17 rows
completed 18 rows
completed 19 rows
completed 20 rows
completed 21 rows
completed 22 rows
completed 23 rows
completed 24 rows
completed 25 rows
completed 26 rows
completed 27 rows
completed 28 rows
completed 29 rows
completed 30 rows
completed 31 rows
completed 32 rows
completed 33 rows


In [43]:
unlabeled_movies_remaining = labeled_movies[(labeled_movies.happy == 0) & (labeled_movies.sad == 0) & (labeled_movies.energetic == 0) & (labeled_movies.calm == 0)]

In [44]:
unlabeled_movies_remaining.shape

(4, 55)

## All that is left is the 4 highly explicit movies that openAI will not take in as input, team decided to drop them

In [ ]:
# Explicit Movies
#labeled_movies[labeled_movies.title == 'Sharp Stick'] # index 11504 NOT DONE
#labeled_movies[labeled_movies.title == 'Beneath the Valley of the Ultra-Vixens'] # index 944 NOT DONE
#labeled_movies[labeled_movies.title == 'Thundercrack!'] # index 470 NOT DONE
#labeled_movies[labeled_movies.title == 'The Private Afternoons of Pamela Mann'] # index 295 NOT DONE

In [45]:
labeled_movies.to_csv('wiki_movies_mood_label_by_gpt-3.5-turbo_final.csv')

In [14]:
labeled_movies_test = pd.read_csv("wiki_movies_mood_label_by_gpt-3.5-turbo_final.csv", low_memory=False)

In [15]:
labeled_movies_test = labeled_movies_test.drop(11504)
labeled_movies_test = labeled_movies_test.drop(944)
labeled_movies_test = labeled_movies_test.drop(470)
labeled_movies_test = labeled_movies_test.drop(295)

In [16]:
labeled_movies_test.shape

(11938, 56)

In [17]:
labeled_movies_test.to_csv('wiki_movies_mood_label_by_gpt-3.5-turbo_final.csv')